In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

In [0]:
df = spark.read.table("workspace.bronze.crm_prd_info")
display(df.printSchema())
display(df.show())

In [0]:
# rename columns
mapping_names = {
    "prd_id": "id",
    "prd_key": "key",
    "prd_nm": "name",
    "prd_cost": "cost",
    "prd_line": "line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}

for old_col, new_col in mapping_names.items():
    df = df.withColumnRenamed(old_col, new_col)

df.printSchema()

In [0]:
# trimming
for col in df.schema.fields:
    if isinstance(col.dataType, StringType):
        df = df.withColumn(col.name, F.trim(col.name))

df.printSchema()

In [0]:
# Standardize line names
df = df.withColumn(
    "line", 
    F.when(df.line == "T", "Touring")
    .when(df.line == "R", "Road")
    .when(df.line == "S", "Sport")
    .when(df.line == "M", "Mountain")
    .otherwise("N/A")
)
df.select(df.line).distinct().show()


In [0]:
# write
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_product_info")

In [0]:
# df.show(40,truncate=False)

In [0]:
# s

In [0]:
# df = df.withColumn("cat_id", F.regexp_replace(F.substring(F.col("key"), 1, 5), "-", "_"))
# df = df.withColumn("key", F.substring(F.col("key"), 7, F.length(F.col("key"))))
# df.show(truncate=False)